In [ ]:
import pandas as pd
import requests
from io import StringIO
import numpy as np

In [ ]:
'''

base_url = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv"

all_data = []

for year in range(2019, 2026):

    offset = 0
    limit = 50000

    while True:
        query = (
            f"crash_date between "
            f"'{year}-01-01T00:00:00' and "
            f"'{year}-12-31T23:59:59'"
        )
        params = {
            "$where": query,
            "$limit": limit,
            "$offset": offset
        }

        response = requests.get(base_url, params=params)
        temp = pd.read_csv(StringIO(response.text))
        
        if temp.empty:
            break

        all_data.append(temp)
        print(f"{year} | offset {offset} | rows: {len(temp)}")

        offset += limit

df = pd.concat(all_data, ignore_index=True)

print(df.shape)
df =pd.read_csv("nyc_crashes_2019_2025.csv")

'''

In [ ]:
df

In [ ]:
print("NAZWY KOLUMN")
print(df.columns.tolist())

print("\nTYPY KOLUMN")
print(df.dtypes)

missing_summary = (
    df.isnull()
    .mean()
    .mul(100)
    .reset_index()
)

missing_summary.columns = ["column", "missing_percent"]

missing_summary = missing_summary.sort_values(
    by="missing_percent",
    ascending=False
)

print("\nBRAKI DANYCH W KOLUMNACH")
print(missing_summary)

In [ ]:
df["crash_date"] = pd.to_datetime(
    df["crash_date"],
    errors="coerce"
)

df["year"] = df["crash_date"].dt.year

missing_matrix = (
    df
    .groupby("year")
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

missing_matrix = missing_matrix.drop(
    columns=["year"],
    errors="ignore"
)

# podział kolumn na pół
midpoint = len(missing_matrix.columns) // 2

missing_matrix_part1 = missing_matrix.iloc[:, :midpoint]
missing_matrix_part2 = missing_matrix.iloc[:, midpoint:]


display(missing_matrix_part1)

display(missing_matrix_part2)

In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Wczytanie granic
boroughs = gpd.read_file("boroughs.geojson")
boroughs = boroughs.to_crs("EPSG:4326")

print(boroughs.columns)
display(boroughs.head())

In [ ]:
#  Testowa próbka 20 000 rekordów z istniejącym borough i GPS
test_borough_geo = (
    df[
        df["borough"].notna()
        & df["latitude"].notna()
        & df["longitude"].notna()
        & (df["latitude"] != 0)
        & (df["longitude"] != 0)
    ]
    .sample(20000, random_state=42)
    .copy()
)

test_borough_geo["original_borough"] = test_borough_geo["borough"]

#   GeoDataFrame
points_test = gpd.GeoDataFrame(
    test_borough_geo,
    geometry=gpd.points_from_xy(
        test_borough_geo["longitude"],
        test_borough_geo["latitude"]
    ),
    crs="EPSG:4326"
)

#  Spatial join
joined_test = gpd.sjoin(
    points_test,
    boroughs[["BoroName", "geometry"]],
    how="left",
    predicate="within"
)

comparison_borough_geo = joined_test[
    [
        "latitude",
        "longitude",
        "original_borough",
        "BoroName"
    ]
].copy()

comparison_borough_geo = comparison_borough_geo.rename(
    columns={"BoroName": "recovered_borough"}
)

comparison_borough_geo["is_correct"] = (
    comparison_borough_geo["original_borough"].str.upper()
    ==
    comparison_borough_geo["recovered_borough"].str.upper()
)

display(comparison_borough_geo)

accuracy = comparison_borough_geo["is_correct"].mean() * 100

print(f"Accuracy: {accuracy:.2f}%")
print(comparison_borough_geo["is_correct"].value_counts())

In [ ]:

# rekordy z brakującym borough i poprawnym GPS
missing_borough_df = df[
    df["borough"].isna()
    & df["latitude"].notna()
    & df["longitude"].notna()
    & (df["latitude"] != 0)
    & (df["longitude"] != 0)
].copy()

print("Brakujące borough:", missing_borough_df.shape)

# GeoDataFrame punktów
missing_points = gpd.GeoDataFrame(
    missing_borough_df,
    geometry=gpd.points_from_xy(
        missing_borough_df["longitude"],
        missing_borough_df["latitude"]
    ),
    crs="EPSG:4326"
)

# spatial join
joined_missing = gpd.sjoin(
    missing_points,
    boroughs[["BoroName", "geometry"]],
    how="left",
    predicate="within"
)

# nowa kolumna
joined_missing = joined_missing.rename(
    columns={"BoroName": "borough_from_coords"}
)

# uzupełnienie braków w oryginalnym df
df.loc[
    joined_missing.index,
    "borough"
] = joined_missing["borough_from_coords"]

# kontrola braków po uzupełnieniu
remaining_missing = df["borough"].isna().sum()

print("Pozostałe braki borough:", remaining_missing)

In [ ]:
location_without_coords = df[
    df["location"].notna()
    & (
        df["latitude"].isna()
        | df["longitude"].isna()
        | (df["latitude"] == 0)
        | (df["longitude"] == 0)
    )
]

print(location_without_coords.shape)

display(
    location_without_coords[
        [
            "location",
            "latitude",
            "longitude",
            "borough",
            "on_street_name"
        ]
    ].head(20)
)

In [ ]:
location_without_coords = df[
    df["location"].notna()
    & (df["location"] != "(0.0, 0.0)")
    & (
        df["latitude"].isna()
        | df["longitude"].isna()
        | (df["latitude"] == 0)
        | (df["longitude"] == 0)
    )
]

print(location_without_coords.shape)

display(
    location_without_coords[
        [
            "location",
            "latitude",
            "longitude",
            "borough",
            "on_street_name"
        ]
    ].head(20)
)

In [ ]:
df = df.drop(columns=["location"])

In [ ]:
df

In [ ]:
df.columns

In [ ]:
df["crash_date"] = pd.to_datetime(
    df["crash_date"],
    errors="coerce"
)

df["year"] = df["crash_date"].dt.year

missing_matrix = (
    df
    .groupby("year")
    .apply(lambda x: x.isna().mean() * 100)
    .round(2)
)

missing_matrix = missing_matrix.drop(
    columns=["year"],
    errors="ignore"
)

# podział kolumn na pół
midpoint = len(missing_matrix.columns) // 2

missing_matrix_part1 = missing_matrix.iloc[:, :midpoint]
missing_matrix_part2 = missing_matrix.iloc[:, midpoint:]


display(missing_matrix_part1)


display(missing_matrix_part2)

In [ ]:
cols_location = [
    "borough",
    "zip_code",
    "latitude",
    "longitude"
]

df = df.dropna(
    subset=cols_location,
    how="all"
)

print(df.shape)

In [ ]:
df = df.copy()

In [ ]:
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")
df["crash_time"] = pd.to_datetime(df["crash_time"], format="%H:%M", errors="coerce")

df["year"] = df["crash_date"].dt.year
df["month"] = df["crash_date"].dt.month
df["month_name"] = df["crash_date"].dt.month_name()
df["day"] = df["crash_date"].dt.day
df["day_of_week"] = df["crash_date"].dt.day_name()
df["day_of_week_number"] = df["crash_date"].dt.dayofweek

df["is_weekend"] = df["day_of_week_number"].isin([5, 6])

df["hour"] = df["crash_time"].dt.hour

df["time_of_day_6h"] = pd.cut(
    df["hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=[
        "Night",
        "Morning",
        "Afternoon",
        "Evening"
    ]
)

In [ ]:
df["quarter"] = df["crash_date"].dt.quarter
df["week_of_year"] = df["crash_date"].dt.isocalendar().week.astype("Int64")
df["is_rush_hour"] = df["hour"].between(7, 9) | df["hour"].between(16, 18)

df["total_injured"] = (
    df["number_of_persons_injured"].fillna(0)
    + df["number_of_pedestrians_injured"].fillna(0)
    + df["number_of_cyclist_injured"].fillna(0)
    + df["number_of_motorist_injured"].fillna(0)
)

df["total_killed"] = (
    df["number_of_persons_killed"].fillna(0)
    + df["number_of_pedestrians_killed"].fillna(0)
    + df["number_of_cyclist_killed"].fillna(0)
    + df["number_of_motorist_killed"].fillna(0)
)

df["has_injuries"] = df["total_injured"] > 0
df["has_fatalities"] = df["total_killed"] > 0

df["severity_level"] = np.select(
    [
        df["total_killed"] > 0,
        df["total_injured"] > 0
    ],
    [
        "Fatal",
        "Injury"
    ],
    default="Property Damage Only"
)

In [ ]:
df["borough"] = (
    df["borough"]
    .str.strip()
    .str.title()
)

In [ ]:
print(sorted(df["borough"].dropna().unique()))

In [ ]:
df.to_csv(
    "nyc_collisions_clean.csv",
    index=False
)

print("Plik zapisany: nyc_collisions_clean.csv")

In [ ]:
print(df.shape)

print(df["crash_date"].min())
print(df["crash_date"].max())

print(df["borough"].nunique())

print(df["contributing_factor_vehicle_1"].nunique())

print(df["vehicle_type_code1"].nunique())

In [ ]:
crashes_by_year = (
    df.groupby("year")
    .size()
    .reset_index(name="crashes")
)

display(crashes_by_year)

In [ ]:
crashes_by_month = (
    df.groupby("month_name")
    .size()
    .reset_index(name="crashes")
)

display(crashes_by_month)

In [ ]:
crashes_by_weekday = (
    df.groupby("day_of_week")
    .size()
    .reset_index(name="crashes")
)

display(crashes_by_weekday)

In [ ]:
crashes_by_hour = (
    df.groupby("hour")
    .size()
    .reset_index(name="crashes")
)

display(crashes_by_hour)

In [ ]:
crashes_by_rush_hour_year = (
    df.groupby(["year", "is_rush_hour"])
    .size()
    .reset_index(name="crashes")
)

crashes_by_rush_hour_year["is_rush_hour"] = (
    crashes_by_rush_hour_year["is_rush_hour"]
    .map({
        True: "Rush Hour",
        False: "Non Rush Hour"
    })
)

crashes_by_rush_hour_year["hours_share"] = (
    crashes_by_rush_hour_year["is_rush_hour"]
    .map({
        "Rush Hour": 6,
        "Non Rush Hour": 18
    })
)

crashes_by_rush_hour_year["crashes_per_hour"] = (
    crashes_by_rush_hour_year["crashes"]
    / crashes_by_rush_hour_year["hours_share"]
)

# suma wszystkich wypadków w każdym roku
total_crashes_year = (
    crashes_by_rush_hour_year
    .groupby("year")["crashes"]
    .transform("sum")
)

# procent wszystkich wypadków w danym roku
crashes_by_rush_hour_year["percent_of_year"] = (
    crashes_by_rush_hour_year["crashes"]
    / total_crashes_year
) * 100


crashes_by_rush_hour_year["percent_of_year"] = (
    crashes_by_rush_hour_year["percent_of_year"]
    .round(2)
)

display(crashes_by_rush_hour_year)

In [ ]:
crashes_by_rush_hour_year.to_csv(
    "crashes_by_rush_hour_year.csv",
    index=False
)

In [ ]:
crashes_by_weekend_year = (
    df.groupby(["year", "is_weekend"])
    .size()
    .reset_index(name="crashes")
)

crashes_by_weekend_year["is_weekend"] = (
    crashes_by_weekend_year["is_weekend"]
    .map({
        True: "Weekend",
        False: "Weekday"
    })
)

# liczba dni
crashes_by_weekend_year["days_share"] = (
    crashes_by_weekend_year["is_weekend"]
    .map({
        "Weekend": 2,
        "Weekday": 5
    })
)

# średnia liczba wypadków na 1 dzień
crashes_by_weekend_year["crashes_per_day"] = (
    crashes_by_weekend_year["crashes"]
    / crashes_by_weekend_year["days_share"]
)

# suma wszystkich wypadków w danym roku
total_crashes_year = (
    crashes_by_weekend_year
    .groupby("year")["crashes"]
    .transform("sum")
)

# procent wszystkich wypadków w roku
crashes_by_weekend_year["percent_of_year"] = (
    crashes_by_weekend_year["crashes"]
    / total_crashes_year
) * 100

# zaokrąglenie
crashes_by_weekend_year["percent_of_year"] = (
    crashes_by_weekend_year["percent_of_year"]
    .round(2)
)

display(crashes_by_weekend_year)

In [ ]:
severity_counts = (
    df["severity_level"]
    .value_counts()
)

print(severity_counts)

In [ ]:
borough_crashes = (
    df.groupby("borough")
    .size()
    .reset_index(name="crashes")
    .sort_values("crashes", ascending=False)
)

display(borough_crashes)

In [ ]:
borough_fatalities = (
    df.groupby("borough")["total_killed"]
    .sum()
    .reset_index()
    .sort_values("total_killed", ascending=False)
)

display(borough_fatalities)

In [ ]:
top_factors = (
    df["contributing_factor_vehicle_1"]
    .value_counts()
    .head(30)
)

print(top_factors)

In [ ]:
fatal_factors = (
    df.groupby("contributing_factor_vehicle_1")["total_killed"]
    .sum()
    .sort_values(ascending=False)
    .head(15)
)

print(fatal_factors)

In [ ]:
vehicle_crashes = (
    df["vehicle_type_code1"]
    .value_counts()
    .head(20)
)

print(vehicle_crashes)

In [ ]:
heatmap_data = (
    df.pivot_table(
        index="day_of_week",
        columns="hour",
        values="collision_id",
        aggfunc="count"
    )
)

display(heatmap_data)

In [ ]:
spatial_df = df[
    df["latitude"].notna()
    & df["longitude"].notna()
    & (df["latitude"] != 0)
    & (df["longitude"] != 0)
].copy()

print(spatial_df.shape)

In [ ]:
pip install plotly

In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(
    spatial_df.sample(10000),
    lat="latitude",
    lon="longitude",
    zoom=9,
    height=700,
    opacity=0.4
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()

In [ ]:
fig = px.density_mapbox(
    spatial_df.sample(50000),
    lat="latitude",
    lon="longitude",
    radius=8,
    zoom=9,
    height=700
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin=dict(l=0, r=0, t=0, b=0)
)

fig.show()

In [ ]:
dangerous_locations = (
    spatial_df
    .groupby(["latitude", "longitude"])
    .size()
    .reset_index(name="crashes")
    .sort_values("crashes", ascending=False)
)

display(dangerous_locations.head(20))

In [ ]:
dangerous_streets = (
    spatial_df["on_street_name"]
    .value_counts()
    .head(20)
)

display(dangerous_streets)

In [ ]:
dangerous_intersections = (
    spatial_df[
        spatial_df["on_street_name"].notna()
        & spatial_df["cross_street_name"].notna()
    ]
    .groupby(
        [
            "on_street_name",
            "cross_street_name"
        ]
    )
    .size()
    .reset_index(name="crashes")
    .sort_values("crashes", ascending=False)
)

display(dangerous_intersections.head(20))

In [ ]:
fatal_hotspots = (
    spatial_df
    .groupby(["latitude", "longitude"])["total_killed"]
    .sum()
    .reset_index()
)

fatal_hotspots = fatal_hotspots[
    fatal_hotspots["total_killed"] > 2
]

fatal_hotspots = fatal_hotspots.sort_values(
    "total_killed",
    ascending=False
)



display(fatal_hotspots.head(20))

In [ ]:
import plotly.express as px

fig = px.scatter_mapbox(
    fatal_hotspots,
    lat="latitude",
    lon="longitude",
    size="total_killed",
    color="total_killed",
    hover_data=["total_killed"],
    zoom=9,
    height=800,
    size_max=30
)

fig.update_layout(
    mapbox_style="carto-positron",
    title="Fatal Crash Hotspots in NYC",
    margin=dict(l=0, r=0, t=50, b=0)
)

fig.show()

In [ ]:
fatal_hotspots.to_csv(
    "fatal_hotspots_nyc.csv",
    index=False
)